YOLO V11 P2

In [6]:
# CODE BLOCK: COMPLETE PREPROCESSING (Corrected for All Folders)

# 1. Install Ultralytics (YOLO)
!pip install ultralytics -q

import os
import shutil
import zipfile
import xml.etree.ElementTree as ET
import numpy as np
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from PIL import Image
from google.colab import drive

# --- CONFIGURATION ---
# If your file is in Drive, we mount it. If you uploaded manually, we skip this.
DRIVE_ZIP_PATH = '/content/drive/MyDrive/FloW_RI_Sequence_1_10.zip'
LOCAL_ZIP_PATH = '/content/FloW_RI_Sequence_1_10.zip'
DATA_DIR = '/content/FloW_RI_Dataset'
OUTPUT_DIR = '/content/trash_dataset_yolo11'

# --- 2. SETUP DATA ---
print("Setting up data...")

# Check if we need to mount drive
if not os.path.exists(LOCAL_ZIP_PATH):
    if not os.path.exists('/content/drive'):
        print("Mounting Google Drive to find dataset...")
        drive.mount('/content/drive')

    if os.path.exists(DRIVE_ZIP_PATH):
        print(f"Found zip in Drive. Copying to local runtime...")
        shutil.copy(DRIVE_ZIP_PATH, LOCAL_ZIP_PATH)
    else:
        print(f"⚠️ WARNING: Could not find {DRIVE_ZIP_PATH}")
        print("Please ensure 'FloW_RI_Sequence_1_10.zip' is uploaded to Colab or your Drive.")

# Unzip
if os.path.exists(LOCAL_ZIP_PATH):
    print("Unzipping dataset...")
    if os.path.exists(DATA_DIR): shutil.rmtree(DATA_DIR)
    os.makedirs(DATA_DIR, exist_ok=True)
    with zipfile.ZipFile(LOCAL_ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(DATA_DIR)
    print("Unzipping complete.")
else:
    raise FileNotFoundError("Dataset zip file not found! Upload it and try again.")

# --- 3. CONVERT TO YOLO FORMAT ---
print("Converting annotations...")

if os.path.exists(OUTPUT_DIR): shutil.rmtree(OUTPUT_DIR)
for p in ['images/train', 'images/val', 'labels/train', 'labels/val']:
    os.makedirs(os.path.join(OUTPUT_DIR, p), exist_ok=True)

def convert_xml(xml_file, w, h):
    try:
        root = ET.parse(xml_file).getroot()
        anns = []
        for obj in root.findall('object'):
            b = obj.find('bndbox')
            xmin = float(b.find('xmin').text)
            ymin = float(b.find('ymin').text)
            xmax = float(b.find('xmax').text)
            ymax = float(b.find('ymax').text)

            # YOLO Format: class x_center y_center width height (normalized 0-1)
            xc = ((xmin + xmax) / 2) / w
            yc = ((ymin + ymax) / 2) / h
            bw = (xmax - xmin) / w
            bh = (ymax - ymin) / h

            # Clip to ensure valid range [0, 1]
            anns.append(f"0 {np.clip(xc,0,1)} {np.clip(yc,0,1)} {np.clip(bw,0,1)} {np.clip(bh,0,1)}")
        return "\n".join(anns)
    except: return ""

# --- NEW LOGIC: FIND ALL PAIRS RECURSIVELY ---
pairs = []
print("Scanning all folders for images and labels...")

for root, dirs, files in os.walk(DATA_DIR):
    # We look for a folder named 'Pic'
    if 'Pic' in dirs:
        pic_dir = os.path.join(root, 'Pic')
        # The annotation folder is usually a sibling named 'Pic_Annotation'
        ann_dir = os.path.join(root, 'Pic_Annotation')

        if os.path.exists(ann_dir):
            # Found a valid pair of folders! Process files inside.
            for f in os.listdir(pic_dir):
                if f.endswith('.jpg'):
                    xml_path = os.path.join(ann_dir, f.replace('.jpg', '.xml'))
                    if os.path.exists(xml_path):
                        # Store the full paths
                        pairs.append((os.path.join(pic_dir, f), xml_path))

print(f"✅ Found {len(pairs)} total image-label pairs.")
# You should see ~2227 here now!

# Split
train_pairs, val_pairs = train_test_split(pairs, test_size=0.2, random_state=42)

# Save function
def save_split(pair_list, split):
    for img_path, xml_path in tqdm(pair_list, desc=f"Processing {split}"):
        # Use the original filename
        filename = os.path.basename(img_path)

        # Copy Image
        dst_img = os.path.join(OUTPUT_DIR, 'images', split, filename)
        shutil.copy(img_path, dst_img)

        # Convert & Save Label
        dst_txt = os.path.join(OUTPUT_DIR, 'labels', split, filename.replace('.jpg', '.txt'))

        with Image.open(img_path) as img:
            width, height = img.size

        yolo_txt = convert_xml(xml_path, width, height)
        with open(dst_txt, 'w') as out:
            out.write(yolo_txt)

save_split(train_pairs, 'train')
save_split(val_pairs, 'val')

# --- 4. CREATE DATASET.YAML ---
yaml_content = f"""
path: {OUTPUT_DIR}
train: images/train
val: images/val

names:
  0: floating_trash
"""

with open(f"{OUTPUT_DIR}/dataset.yaml", 'w') as f:
    f.write(yaml_content)

print("\n✅ SUCCESS: Preprocessing complete.")
print(f"Dataset ready at: {OUTPUT_DIR}/dataset.yaml")

Setting up data...
Unzipping dataset...
Unzipping complete.
Converting annotations...
Scanning all folders for images and labels...
✅ Found 2227 total image-label pairs.


Processing train:   0%|          | 0/1781 [00:00<?, ?it/s]

Processing val:   0%|          | 0/446 [00:00<?, ?it/s]


✅ SUCCESS: Preprocessing complete.
Dataset ready at: /content/trash_dataset_yolo11/dataset.yaml


In [7]:
# --------------------------------------------------------------------------------
# ROBUST TRAINING SCRIPT: YOLO11-P2 (Tiny Object Optimized) + AUTO-RESUME
# --------------------------------------------------------------------------------

import os
import shutil
import yaml
from ultralytics import YOLO
from google.colab import drive

# --- 1. CONFIGURATION ---
DATASET_YAML = '/content/trash_dataset_yolo11/dataset.yaml'
# This is the folder in your Drive where checkpoints will be saved/loaded
DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/YOLO11_Trash_Project/checkpoints'
RUN_NAME = 'yolo11_p2_highres'

# Training Parameters
EPOCHS = 50
IMG_SIZE = 1024
BATCH_SIZE = 4

# --- 2. SETUP & CHECKS ---
# Mount Drive
if not os.path.exists('/content/drive'):
    print("Mounting Google Drive...")
    drive.mount('/content/drive')

# Create Drive directory if it doesn't exist
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)

# Check if dataset exists
if not os.path.exists(DATASET_YAML):
    raise FileNotFoundError(f"Dataset not found at {DATASET_YAML}. Please run the Preprocessing step first!")

# Define paths for resume check
drive_last_pt = os.path.join(DRIVE_CHECKPOINT_DIR, 'last.pt')
drive_best_pt = os.path.join(DRIVE_CHECKPOINT_DIR, 'best.pt')

# --- 3. DEFINE CUSTOM P2 ARCHITECTURE ---
p2_config_str = """
# Ultralytics YOLO11-P2 for Tiny Object Detection
nc: 1
scales:
  s: [0.50, 0.50, 1024]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 2, C3k2, [256, False, 0.25]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 2, C3k2, [512, False, 0.25]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 2, C3k2, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 2, C3k2, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]

head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]

  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 2, C3k2, [256, False]]

  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 2], 1, Concat, [1]]
  - [-1, 2, C3k2, [128, False]]  # P2/4 (TINY HEAD)

  - [-1, 1, Conv, [128, 3, 2]]
  - [[-1, 15], 1, Concat, [1]]
  - [-1, 2, C3k2, [256, False]]

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 12], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 9], 1, Concat, [1]]
  - [-1, 2, C3k2, [1024, True]]

  - [[18, 21, 24, 27], 1, Detect, [nc]]
"""
custom_yaml_path = "/content/yolo11-p2.yaml"
with open(custom_yaml_path, "w") as f:
    f.write(p2_config_str)

# --- 4. DEFINE AUTO-SAVE CALLBACK ---
def save_to_drive_callback(trainer):
    """Copies weights to Drive at the end of every epoch."""
    # Paths from the trainer object
    local_last = trainer.last
    local_best = trainer.best

    if local_last and os.path.exists(local_last):
        shutil.copy(local_last, drive_last_pt)

    if local_best and os.path.exists(local_best):
        shutil.copy(local_best, drive_best_pt)

    print(f"💾 Auto-saved checkpoints to Drive: {DRIVE_CHECKPOINT_DIR}")

# --- 5. INITIALIZE MODEL (Resume or Fresh Start) ---

if os.path.exists(drive_last_pt):
    print(f"🔄 Found existing checkpoint in Drive: {drive_last_pt}")
    print("RESUMING TRAINING from where it left off...")

    # Load the existing model state
    model = YOLO(drive_last_pt)

    # Attach the callback
    model.add_callback("on_train_epoch_end", save_to_drive_callback)

    # Resume training
    results = model.train(resume=True)

else:
    print("✨ No checkpoint found. Starting FRESH training...")
    print(f"   - Architecture: YOLO11-P2")
    print(f"   - Image Size: {IMG_SIZE}")

    # Initialize fresh model
    model = YOLO(custom_yaml_path)

    # Load pretrained weights (transfer learning)
    try:
        model.load("yolo11s.pt")
    except:
        print("Note: Standard weights loaded partially (expected due to P2 architecture).")

    # Attach the callback
    model.add_callback("on_train_epoch_end", save_to_drive_callback)

    # Start training
    results = model.train(
        data=DATASET_YAML,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        name=RUN_NAME,
        patience=10,
        augment=True,
        lr0=0.01,
        cos_lr=True,
        project='/content/runs/detect'
    )

print("\n✅ Training Process Complete!")
print(f"Final model is safely stored at: {drive_best_pt}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✨ No checkpoint found. Starting FRESH training...
   - Architecture: YOLO11-P2
   - Image Size: 1024
WARNING ⚠️ no model scale passed. Assuming scale='s'.
Transferred 198/551 items from pretrained weights
Ultralytics 8.3.232 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/trash_dataset_yolo11/dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=

In [ ]:
# CODE BLOCK: FINAL EVALUATION (YOLO11-P2 High-Res)

from ultralytics import YOLO
import os

# 2. Define Paths
# Priority 1: Local training output
local_weights = '/content/runs/detect/yolo11_p2_highres/weights/best.pt'
# Priority 2: Drive backup
drive_weights = '/content/drive/MyDrive/YOLO11_Trash_Project/checkpoints/best.pt'

if os.path.exists(local_weights):
    model_path = local_weights
    print(f"✅ Found local weights: {model_path}")
elif os.path.exists(drive_weights):
    model_path = drive_weights
    print(f"✅ Found backup weights in Drive: {model_path}")
else:
    raise FileNotFoundError("❌ Could not find 'best.pt'. Did the P2 training finish?")

# 3. Load the Model
print("Loading YOLO11-P2 model...")
model = YOLO(model_path)

# 4. Run Validation
print("\nRunning final validation...")
# Note: We use imgsz=1024 because this model was trained at High Res
metrics = model.val(split='val', imgsz=1024)

# 5. Print Final Report
print("\n" + "="*40)
print("🏆 FINAL RESULTS: YOLO11-P2 (Tiny Object Optimized)")
print("="*40)
print(f"mAP50 (Standard Accuracy):   {metrics.box.map50:.4f}")
print(f"mAP50-95 (Strict Accuracy):  {metrics.box.map:.4f}")
print(f"Precision:                   {metrics.box.mp:.4f}")
print(f"Recall:                      {metrics.box.mr:.4f}")
print("="*40)

✅ Found local weights: /content/runs/detect/yolo11_p2_highres/weights/best.pt
Loading YOLO11-P2 model...

Running final validation...
Ultralytics 8.3.232 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11-p2 summary (fused): 109 layers, 8,571,220 parameters, 0 gradients, 27.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2330.7±789.5 MB/s, size: 237.9 KB)
val: Scanning /content/trash_dataset_yolo11/labels/val.cache... 446 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 446/446 802.9Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 28/28 1.9it/s 14.5s
                   all        446       1069      0.918      0.918      0.955       0.57
Speed: 4.2ms preprocess, 19.5ms inference, 0.0ms loss, 1.7ms postprocess per image
Results saved to /content/runs/detect/val

🏆 FINAL RESULTS: YOLO11-P2 (Tiny Object Optimized)
mAP50 (Standard Accuracy):   0.9553
mAP50-95 (Strict Accuracy):  0.5705
Pr

In [ ]:
# CODE BLOCK: GRADIO APP FOR YOLO11-P2 (High Res)

import gradio as gr
from ultralytics import YOLO
from PIL import Image
import numpy as np
import os

# 2. Load the P2 Model
# Priority 1: Local training output (if session is still active)
local_path = '/content/runs/detect/yolo11_p2_highres/weights/best.pt'
# Priority 2: Drive backup (if you trained earlier)
drive_path = '/content/drive/MyDrive/YOLO11_Trash_Project/checkpoints/best.pt'

if os.path.exists(local_path):
    print(f"✅ Loading local P2 model: {local_path}")
    model = YOLO(local_path)
elif os.path.exists(drive_path):
    print(f"✅ Loading P2 model from Drive: {drive_path}")
    model = YOLO(drive_path)
else:
    print("⚠️ Warning: P2 Model not found. Loading standard YOLO11n (Performance will be worse).")
    model = YOLO('yolo11n.pt')

# 3. Define Detection Function
def detect_trash_p2(input_image, conf_threshold, iou_threshold):
    if input_image is None:
        return None, "Please upload an image."

    # Run YOLO inference at HIGH RESOLUTION (1024)
    # This is critical for the P2 model to work correctly on tiny objects
    results = model.predict(
        source=input_image,
        conf=conf_threshold,
        iou=iou_threshold,
        imgsz=1024  # Match training resolution
    )

    first_result = results[0]

    # Plot results
    plotted_image_bgr = first_result.plot()
    plotted_image_rgb = plotted_image_bgr[..., ::-1]

    # Summary
    trash_count = len(first_result.boxes)
    info_text = f"Detected {trash_count} object(s) using High-Res P2 Model."

    return plotted_image_rgb, info_text

# 4. Create UI
demo = gr.Interface(
    fn=detect_trash_p2,
    inputs=[
        gr.Image(label="Upload Image", type="numpy"),
        gr.Slider(0.1, 1.0, 0.20, label="Confidence (Start low: 0.20)"),
        gr.Slider(0.1, 1.0, 0.45, label="IoU Threshold"),
    ],
    outputs=[
        gr.Image(label="High-Res Detection"),
        gr.Textbox(label="Status")
    ],
    title="🌊 Tiny Trash Detector (YOLO11-P2 High-Res)",
    description="This app uses your custom YOLO11-P2 model running at 1024px resolution to find the smallest floating debris."
)

print("Launching P2 Frontend...")
demo.launch(share=True, debug=True)

✅ Loading local P2 model: /content/runs/detect/yolo11_p2_highres/weights/best.pt
Launching P2 Frontend...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f0c620018d011a4532.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



0: 576x1024 1 floating_trash, 61.8ms
Speed: 4.2ms preprocess, 61.8ms inference, 3.2ms postprocess per image at shape (1, 3, 576, 1024)

0: 576x1024 4 floating_trashs, 28.2ms
Speed: 9.8ms preprocess, 28.2ms inference, 2.1ms postprocess per image at shape (1, 3, 576, 1024)
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://f0c620018d011a4532.gradio.live
